# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We'll display all record sets, their `@id`, and contain fields and columns, all referenced by their `@id`.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet '@id': {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    print(f"  Fields:")
    for field in rs['fields']:
        print(f"    Field '@id': {field['@id']}, DataType: {field.get('dataType','N/A')}, Name: {field.get('name','N/A')}")
    if 'columns' in rs:
        print(f"  Columns:")
        for col in rs['columns']:
            print(f"    Column '@id': {col['@id']}, Field: {col['field']}, Name: {col.get('name','N/A')}")
    print("")  # Empty line for clarity

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s as found above. Each record set's data is referenced by its `@id`.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
print(f"Record set ids: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for RecordSet '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Loaded {len(dataframes[record_set_id])} records.")
            print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        else:
            print("  No records found.")
    except Exception as e:
        print(f"  Error loading records: {e}")
    print("\n")
# Select the first record set for exploration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Displaying head for DataFrame from RecordSet '@id': {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, grouping the data by categorical variables, etc.

Below, we:
* Select a numeric field from the main record set using its `@id`.
* Filter the DataFrame by a threshold value for that field.
* Normalize its distribution.
* Optionally, group results by a categorical field if present.

In [ ]:
# Example EDA on main_record_set_id
df = dataframes.get(main_record_set_id)
if df is not None:
    # Try to auto-detect a numeric field by data type
    numeric_field_id = None
    for rs in dataset.metadata.record_sets:
        if rs['@id'] == main_record_set_id:
            for f in rs['fields']:
                # Assume Float/Integer are numeric
                typ = f.get('dataType','') or ''
                if ('Float' in typ or 'Integer' in typ or 'Number' in typ) and f['@id'] in df.columns:
                    numeric_field_id = f['@id']
                    break
    if not numeric_field_id:
        # Fallback: Try to find first column that looks numeric
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    print(f"Using numeric field for analysis: {numeric_field_id}")

    # Set a threshold for filtering
    threshold = df[numeric_field_id].mean() if numeric_field_id else None

    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize column
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to group by a non-numeric field
        group_field = None
        for col in df.columns:
            if col != numeric_field_id and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field} (showing means):")
            display(grouped_df.head())

## 5. Visualization
Visualize field distributions or relationships between fields.

In [ ]:
# Plot distribution of the selected numeric field
if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# Optionally, show a scatter plot with a second numeric field if one exists
second_numeric_field = None
if df is not None and numeric_field_id:
    for col in df.columns:
        if col != numeric_field_id:
            if pd.api.types.is_numeric_dtype(df[col]):
                second_numeric_field = col
                break
if second_numeric_field:
    plt.figure(figsize=(6,6))
    plt.scatter(df[numeric_field_id], df[second_numeric_field], alpha=0.6)
    plt.xlabel(numeric_field_id)
    plt.ylabel(second_numeric_field)
    plt.title(f"Scatter: {numeric_field_id} vs {second_numeric_field}")
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library. We referenced all entities by their `@id`, loaded and inspected record sets, fields, and columns, performed basic EDA including filtering and normalization, and visualized data distributions.

Further analysis can be performed as appropriate for downstream applications, such as protein bioinformatics or statistical comparison between experimental groups.